In [49]:

import numpy as np

import pandas as pd
from sqlalchemy import text, bindparam

import nhs_waiting_lists as nhs


In [50]:

start_period = "2024-08-01"
end_period = "2025-12"

# PROVIDER_CODES = ['R0B', 'RAJ', 'RTH', 'RTE', "RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW", "RDU", "RH8",
#                   "RWY",
#                   "RXC", "RL4", "RDE", "RXK", "RXR", "RJ2", "RN5", "RHU", "RGN", "RWP", "RWD", "RAJ", 'RTF', 'RXC']
# # PROVIDER_COEDS = ('RAJ', 'RTH', 'RJZ', 'RH5', 'R0B', 'RTF', 'RXC')
# TREATMENT_CODES = (
#     'C_100',
#     'C_101',
#     'C_110',
#     'C_320',
#     'C_330',
#     'C_400',
#     'C_502',
#     'C_301'
# )

PROVIDER_CODES = ["RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW", "RDU", "RH8", "RWY", "RXC", "RL4", "RDE",
                  "RXK", "RXR", "RJ2", "RN5", "RHU", "RGN", "RWP", "RWD", "RAJ"]
TREATMENT_CODES = ('C_101', 'C_110', 'C_320', 'C_330', 'C_400', 'C_502', 'C_301', 'C_100', 'C_101', 'C_110', 'C_120',
                   'C_130', 'C_140', 'C_150', 'C_160', 'C_170', 'C_300', 'C_301', 'C_320', 'C_330', 'C_340', 'C_400',
                   'C_410', 'C_430', 'C_502')

# TREATMENT_CODES = ('C_999',)

provider_code = "RAJ"
treatment_code = "C_110"

consolidated_df = nhs.get_consolidated_df(
    start_period,
    end_period,
    PROVIDER_CODES,
    TREATMENT_CODES,
)

valid = (consolidated_df['incomplete_prev'] > 100) & (consolidated_df['new_periods'] > 30)
consolidated_df = consolidated_df.loc[valid].copy()
consolidated_df

,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-09-01,R0B,C_100,-82.0,1727,4902,4901.0,1.0,1644,1644,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,6628.0,4984.0
1,2024-09-01,R0B,C_101,-194.0,1559,6009,5938.0,71.0,1294,1294,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,7497.0,6203.0
2,2024-09-01,R0B,C_110,-106.0,1535,8598,8732.0,-134.0,1563,1563,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,10267.0,8704.0
3,2024-09-01,R0B,C_120,-156.0,1272,4351,4504.0,-153.0,1269,1269,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,5776.0,4507.0
4,2024-09-01,R0B,C_130,-306.0,2504,8749,8841.0,-92.0,2290,2290,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,11345.0,9055.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4576,2025-09-01,RXR,C_320,-183.0,567,3416,3600.0,-184.0,568,568,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,4167.0,3599.0
4577,2025-09-01,RXR,C_330,23.0,541,2784,2671.0,113.0,451,451,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,3212.0,2761.0
4578,2025-09-01,RXR,C_340,-73.0,349,720,798.0,-78.0,354,354,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,1147.0,793.0
4579,2025-09-01,RXR,C_410,-2.0,200,945,929.0,16.0,182,182,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,1129.0,947.0


In [51]:


# After loading the dataframe
consolidated_df['r_peer_median'] = consolidated_df.groupby(
    ['period', 'treatment']
)['r'].transform('median')

consolidated_df['r_peer_mean'] = consolidated_df.groupby(
    ['period', 'treatment']
)['r'].transform('mean')

consolidated_df['r_peer_std'] = consolidated_df.groupby(
    ['period', 'treatment']
)['r'].transform('std')

group = consolidated_df.groupby(['period', 'treatment'])
consolidated_df['r_mad'] = group['r'].transform(lambda x: 1.4826 * (x - x.median()).abs().median())
consolidated_df['r_z'] = (consolidated_df['r'] - consolidated_df['r_peer_median']) / consolidated_df['r_mad']

consolidated_df['excess_flag'] = (consolidated_df['r_z'] < -2).astype(int)
consolidated_df['cum_excess'] = consolidated_df.groupby(['provider', 'treatment'])['excess_flag'].cumsum()

consolidated_df['denom'] = consolidated_df['incomplete_prev'] + consolidated_df['new_periods']
consolidated_df['r_low'] = consolidated_df['r_peer_median'] - 2 * consolidated_df['r_mad']
consolidated_df['expected_min_resid'] = consolidated_df['r_low'] * consolidated_df['denom']

# Missing patients this month (beyond expected min)
consolidated_df['excess_missing'] = np.where(consolidated_df['residual'] < consolidated_df['expected_min_resid'],
                                             consolidated_df['expected_min_resid'] - consolidated_df['residual'],
                                             0)

consolidated_df

KeyError: 'Column not found: r'

In [ ]:
df_xs = consolidated_df.query("excess_missing > 0")

print(df_xs["excess_missing"].sum())

consolidated_df.query("provider == 'RXR' and treatment == 'C_110' and period == '2024-10'")[[
    'incomplete_prev', 'incomplete', 'new_periods', 'admitted', 'nonadmitted', 'residual', 'excess_missing'
]]


In [ ]:
provider_treatment_rank = (
    consolidated_df
    .groupby(['provider', 'treatment'], as_index=False)['excess_missing']
    .sum()
    .rename(columns={'excess_missing': 'total_excess_missing'})
)

# rank within treatment, or overall
provider_treatment_rank['rank_within_treatment'] = (
    provider_treatment_rank.groupby('treatment')['total_excess_missing']
    .rank(ascending=False, method='min')
)
provider_treatment_rank['rank_overall'] = provider_treatment_rank['total_excess_missing'] \
    .rank(ascending=False, method='min')

provider_treatment_rank = provider_treatment_rank.sort_values('total_excess_missing', ascending=False)

provider_treatment_rank.head(10)

In [ ]:
provider_rank = (
    consolidated_df
    .groupby(['provider'], as_index=False)['excess_missing']
    .sum()
    .rename(columns={'excess_missing': 'total_excess_missing'})
)
provider_rank = provider_rank.sort_values('total_excess_missing', ascending=False)

provider_rank.head(10)

In [ ]:
import matplotlib.pyplot as plt

top20 = provider_treatment_rank.head(20)
plt.figure(figsize=(10, 6))
plt.barh(top20['provider'], top20['total_excess_missing'])
plt.gca().invert_yaxis()
plt.xlabel('Excess missing patients (beyond peer norm)')
plt.title('Top-20 providers by cumulative “dark backlog”')
plt.tight_layout()
plt.show()

In [ ]:

from plotnine import (
    ggplot, aes, geom_histogram, geom_vline, labs
)

quantile_05 = consolidated_df["r_peer_median"].quantile(0.05)

apple_returns_figure = (
        ggplot(consolidated_df, aes(x="r_peer_median"))
        + geom_histogram(bins=100)
        + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
        + labs(x="", y="", title="Distribution of daily Apple stock returns")

)
apple_returns_figure.show()

In [ ]:

from plotnine import (
    ggplot, aes, geom_histogram, geom_vline, labs
)

quantile_05 = consolidated_df["r"].quantile(0.05)

apple_returns_figure = (
        ggplot(consolidated_df, aes(x="r"))
        + geom_histogram(bins=100)
        + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
        + labs(x="", y="", title="Distribution of daily Apple stock returns")

)
apple_returns_figure.show()

In [ ]:
from plotnine import (
    theme_minimal,
    scale_x_datetime
)
from plotnine import ggplot, aes, geom_line, facet_wrap, labs, theme_bw

p7 = (
        ggplot(consolidated_df, aes(x="period", y="incomplete", color="treatment", group="treatment"))
        + geom_line()
        + facet_wrap("~provider", ncol=2, scales="free_y")  # one plot per provider
        + labs(
    title="Incomplete Pathways by Specialty and Provider",
    x="Month",
    y="Total incomplete pathways",
    color="Specialty Code"
)
        + theme_bw(base_size=9)
)
p7

In [ ]:
p = (
        ggplot(consolidated_df.query(f"provider == '{provider_code}'"),
               aes(x="period", y="incomplete", color="treatment", group="treatment"))
        + geom_line()
        + theme_minimal()
        + scale_x_datetime(date_labels="%y-%m")
)
p

In [ ]:
p2 = (
        ggplot(consolidated_df.query(f"provider == '{provider_code}'").query("treatment == 'C_320'"),
               aes(x="period", y="incomplete", color="treatment", group="treatment"))
        + geom_line()
        + theme_minimal()
        + scale_x_datetime(date_labels="%y-%m")
)
p2

In [ ]:

import statsmodels.api as sm

df = consolidated_df.query(f"provider == '{provider_code}'").query("treatment == 'C_320'")
X = sm.add_constant(df['r_peer_median'])
y = df['r']
ols = sm.OLS(y, X).fit(cov_type='HC3')
print(ols.params, ols.rsquared)
print(ols.summary2())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# # Convert to datetime if not already
# subset_c_400 = consolidated_df[(consolidated_df["provider"] == "RAJ") & (consolidated_df["treatment"] == "C_400")].copy()
# subset_c_400["period"] = pd.to_datetime(subset_c_400["period"], errors="coerce")

consolidated_df = consolidated_df.reset_index()

returns_wide = (
    consolidated_df[(consolidated_df["provider"] == provider_code)].query(
        'treatment == "C_400" or treatment=="C_320"')
    .pivot(
        index="period",
        columns="treatment",
        values="residual"
    )
    .reset_index()
)
returns_wide

In [ ]:


# Plot
fig, ax = plt.subplots(
    figsize=(10, 15),
    sharex=True,
    sharey=False,
    nrows=3,
    ncols=1,
)
returns_wide["period"] = pd.to_datetime(returns_wide["period"], errors="coerce")


# subset_c_400 = df[(df["provider"] == "RAJ") & (df["treatment"] == "C_400")].copy()
# subset_c_400["period"] = pd.to_datetime(subset_c_400["period"], errors="coerce")

def align_yaxis_zero(ax1, ax2):
    """Align the zero points of two y-axes."""
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = ax2.get_ylim()

    # Calculate the ratio of positive to negative range for ax1
    if y1_min < 0 < y1_max:
        ratio1 = y1_max / abs(y1_min)
    else:
        return  # Don't adjust if ax1 doesn't cross zero

    # Apply same ratio to ax2 if it crosses zero
    if y2_min < 0 < y2_max:
        # Adjust limits to match ratio
        new_y2_max = abs(y2_min) * ratio1
        ax2.set_ylim(y2_min, new_y2_max)


for ax_idx, colname in ([0, "C_320"], [1, "C_400"], [2, "C_110"]):
    ax[ax_idx].grid(True, alpha=0.3)

    subset = consolidated_df.query(f"provider == '{provider_code}' and treatment == @colname")

    ax[ax_idx].plot(subset["period"], subset["incomplete"], lw=1.5)
    ax[ax_idx].set_title(f"Residual Flow {colname}")
    ax[ax_idx].set_ylabel("Residual patients")
    ax[ax_idx].grid(True, alpha=0.3)
    ax[ax_idx].fill_between(subset["period"], subset["r"], where=subset["r"] < 0, facecolor='green', alpha=.5)
    ax[ax_idx].plot(subset["period"], subset["residual"], lw=1.5)

    ax2 = ax[ax_idx].twinx()
    ax2.plot(subset["period"], subset["r"], lw=1.5)
    ax2.fill_between(subset["period"], subset["r"], where=subset["r"] < 0, facecolor='green', alpha=.5)
    align_yaxis_zero(ax[ax_idx], ax2)

    # Yearly major ticks
    ax[ax_idx].xaxis.set_major_locator(mdates.YearLocator())
    ax[ax_idx].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # # Optional: quarterly minor ticks
    # ax[0].xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    # ax[0].xaxis.set_minor_formatter(mdates.DateFormatter("%b"))

    plt.setp(ax[ax_idx].get_xticklabels(), rotation=0, ha="center")
plt.tight_layout()
plt.show()


In [ ]:

treatments = subset["treatment"].unique()
ncols = 3
nrows = -(-len(treatments) // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), sharex=True)
axes = axes.flatten()

for ax, tcode in zip(axes, treatments):
    d = subset[subset["treatment"] == tcode]
    d = d.sort_values("period")

    ax.xaxis.set_major_locator(mdates.YearLocator())  # one tick per year
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis="x", rotation=0)

    d_norm = d[["admitted", "nonadmitted", "new_periods"]].div(
        d[["admitted", "nonadmitted", "new_periods"]].sum(axis=1), axis=0)

    ax.stackplot(
        d["period"],
        d_norm["new_periods"],
        d_norm["admitted"],
        d_norm["nonadmitted"],
        # labels=["new_periods", "Admitted", "Non-admitted"],
        alpha=0.8,
    )
    ax.set_title(tcode)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)

# tidy up
for ax in axes[len(treatments):]:
    ax.set_visible(False)

fig.suptitle("RAJ – RTT Flows by Specialty", fontsize=16)
fig.tight_layout()
fig.subplots_adjust(top=0.93)

# one global legend
fig.legend(["New periods", "Admitted", "Non-admitted"],
           loc="lower right", bbox_to_anchor=(0.98, 0.95))

plt.show()
